# Лабораторна робота 3

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import copy
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '  device:', device)

# Функції для навчання

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = loss_fn(out, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)

    return total_loss / n, correct / n


def fit(model, train_dl, val_dl, epochs, lr, weight_decay=0.0, patience=None, verbose=1):
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    history, best_loss, best_state, wait = [], np.inf, None, 0
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_dl, loss_fn, optimizer)
        va_loss, va_acc = run_epoch(model, val_dl, loss_fn)
        history.append({'epoch': epoch, 'train_loss': tr_loss, 'val_loss': va_loss,
                        'train_acc': tr_acc, 'val_acc': va_acc,
                        'time': time.perf_counter() - start})

        if verbose and (epoch == 1 or epoch % verbose == 0):
            print(f'epoch {epoch:3d}  train loss {tr_loss:.4f}  val loss {va_loss:.4f}  '
                  f'train acc {tr_acc:.4f}  val acc {va_acc:.4f}  ({history[-1]["time"]:.1f} s)')

        if va_loss < best_loss:
            best_loss, best_state, wait = va_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if patience is not None and wait >= patience:
                break

    if patience is not None:
        model.load_state_dict(best_state)

    return pd.DataFrame(history).set_index('epoch')


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, ys = [], []
    for xb, yb in loader:
        probs.append(torch.softmax(model(xb.to(device)), dim=1).cpu())
        ys.append(yb)
    probs = torch.cat(probs).numpy()
    return probs.argmax(1), probs, torch.cat(ys).numpy()


def to_loader(X, y, batch_size, shuffle=False):
    X = np.asarray(X)
    X_t = torch.tensor(X, dtype=torch.float32 if X.dtype.kind == 'f' else torch.long)
    y_t = torch.tensor(np.asarray(y), dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def plot_history(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    colors = sns.color_palette('tab10')

    for (name, h), color in zip(histories.items(), colors):
        axes[0].plot(h.index, h['train_loss'], color=color, linestyle='--', label=f'{name}: train')
        axes[0].plot(h.index, h['val_loss'], color=color, label=f'{name}: validation')
        axes[1].plot(h.index, h['train_acc'], color=color, linestyle='--', label=f'{name}: train')
        axes[1].plot(h.index, h['val_acc'], color=color, label=f'{name}: validation')

    axes[0].set_ylabel('Loss')
    axes[1].set_ylabel('Accuracy')
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def history_summary(histories):
    return pd.DataFrame([{
        'Model': name,
        'Epochs': len(h),
        'Best epoch': h['val_loss'].idxmin(),
        'Min val loss': h['val_loss'].min(),
        'Final train loss': h['train_loss'].iloc[-1],
        'Final val loss': h['val_loss'].iloc[-1],
        'Time / epoch, s': h['time'].iloc[-1] / len(h),
        'Total time, s': h['time'].iloc[-1],
    } for name, h in histories.items()]).set_index('Model')

# Завдання 1. Повнозв'язана нейронна мережа (MLP)

## 1.1 Підготовка даних

In [ ]:
df = pd.read_csv('../Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')

df.columns = df.columns.str.strip()
df['Label'] = df['Label'].str.strip().str.replace('\ufffd', '-', regex=False)

TARGET = 'Label'
features = [c for c in df.columns if c != TARGET]

num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

const_cols = [c for c in features if df[c].nunique() == 1]
df = df.drop(columns=const_cols)

df['Label_bin'] = (df[TARGET] != 'BENIGN').astype(int)

attacks = df[df['Label_bin'] == 1]
benign = df[df['Label_bin'] == 0].sample(n=len(attacks), random_state=42)
data = pd.concat([attacks, benign]).sample(frac=1, random_state=42).reset_index(drop=True)

X = data.drop(columns=[TARGET, 'Label_bin'])
y = data['Label_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

scaler = StandardScaler().fit(X_train)
X_train_sc, X_test_sc = scaler.transform(X_train), scaler.transform(X_test)
X_tr_sc, X_val_sc = scaler.transform(X_tr), scaler.transform(X_val)

print('Train:', X_tr_sc.shape, '  Validation:', X_val_sc.shape, '  Test:', X_test_sc.shape)

In [ ]:
tab_train_dl = to_loader(X_tr_sc, y_tr, batch_size=64, shuffle=True)
tab_val_dl = to_loader(X_val_sc, y_val, batch_size=512)
tab_test_dl = to_loader(X_test_sc, y_test, batch_size=512)

## 1.2 Архітектура MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, hidden, dropout=0.0, n_out=2):
        super().__init__()
        layers, prev = [], n_in
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


n_features = X_tr_sc.shape[1]
print(MLP(n_features, hidden=(128, 64), dropout=0.3))

## 1.3 Навчання

Дві моделі: велика мережа без регуляризації і менша з dropout, weight decay та early stopping.

In [ ]:
torch.manual_seed(42)
mlp_plain = MLP(n_features, hidden=(256, 128, 64), dropout=0.0)
print('Parameters:', count_params(mlp_plain))

hist_plain = fit(mlp_plain, tab_train_dl, tab_val_dl, epochs=200, lr=1e-3, verbose=25)

In [ ]:
torch.manual_seed(42)
mlp_reg = MLP(n_features, hidden=(128, 64), dropout=0.3)
print('Parameters:', count_params(mlp_reg))

hist_reg = fit(mlp_reg, tab_train_dl, tab_val_dl, epochs=200, lr=1e-3,
               weight_decay=1e-4, patience=20, verbose=25)

## 1.4 Learning curves

In [ ]:
mlp_histories = {'MLP (no regularization)': hist_plain, 'MLP (dropout + early stopping)': hist_reg}

plot_history(mlp_histories, 'MLP learning curves')
print(history_summary(mlp_histories).round(4).T)

## 1.5 Оцінка на тестовій вибірці та порівняння з ЛР1

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

classic = {
    'kNN (k = 1)': (KNeighborsClassifier(n_neighbors=1), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=10, random_state=42), False),
    'SVM (C = 100, gamma = 1)': (SVC(kernel='rbf', C=100, gamma=1, random_state=42), True),
    'Random Forest': (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1), False),
    'AdaBoost': (AdaBoostClassifier(n_estimators=100, random_state=42), False),
}

def tab_row(name, p, fit_time):
    return {'Model': name,
            'Accuracy': accuracy_score(y_test, p),
            'Precision': precision_score(y_test, p),
            'Recall': recall_score(y_test, p),
            'F1': f1_score(y_test, p),
            'Errors': int((p != y_test.values).sum()),
            'Fit, s': fit_time}

tab_rows, tab_preds = [], {}

for name, (model, scaled) in classic.items():
    X_fit, X_eval = (X_train_sc, X_test_sc) if scaled else (X_train, X_test)
    start = time.perf_counter()
    model.fit(X_fit, y_train)
    fit_time = time.perf_counter() - start
    tab_preds[name] = model.predict(X_eval)
    tab_rows.append(tab_row(name, tab_preds[name], fit_time))

for name, model, h in [('MLP (no regularization)', mlp_plain, hist_plain),
                       ('MLP (dropout + early stopping)', mlp_reg, hist_reg)]:
    tab_preds[name] = predict(model, tab_test_dl)[0]
    tab_rows.append(tab_row(name, tab_preds[name], h['time'].iloc[-1]))

tab_scores = pd.DataFrame(tab_rows).set_index('Model').sort_values('F1', ascending=False)
print(tab_scores.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, ['MLP (dropout + early stopping)', 'MLP (no regularization)', 'AdaBoost']):
    sns.heatmap(confusion_matrix(y_test, tab_preds[name]), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['BENIGN', 'Attack'], yticklabels=['BENIGN', 'Attack'], ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

fig.suptitle('Confusion matrices (test set)')
plt.tight_layout()
plt.show()

### Висновок

MLP досягла F1 = 0.991 (12 помилок з 1308) — на рівні SVM і kNN з ЛР1, але трохи гірше за дерева: AdaBoost — 0.996 (5 помилок), Decision Tree і Random Forest — 0.995. На табличних даних з кількома сильними ознаками дерева працюють краще й навчаються в десятки разів швидше (0.02–0.4 с проти 2–9 с).

Сильного перенавчання немає: train і validation loss майже збігаються (≈0.016 і ≈0.013 наприкінці). Велика мережа без регуляризації досягла мінімуму validation loss уже на 25-й епосі (0.011), далі він лише коливався зі стрибками. Мережа з dropout і early stopping зупинилась на 63-й епосі з таким самим validation loss, але навчалась у 4 рази швидше. Задача проста — атаки легко відділяються, тому мережі нема чого «запам'ятовувати».

# Завдання 2. Згорткові нейронні мережі

**Датасет:** Pokemon Images and Types — 809 зображень покемонів 120×120 (PNG з прозорим фоном), мітка — основний тип (`Type1`).

## 2.1 Опис датасету

In [ ]:
from PIL import Image
from torchvision import transforms, models

poke = pd.read_csv('pokemon.csv')

print('Images:', len(poke), '  types:', poke['Type1'].nunique())
print()
print(poke['Type1'].value_counts().to_string())

In [ ]:
CLASSES = ['Water', 'Grass', 'Fire', 'Electric']
IMG_SIZE = 128

def load_image(name):
    img = Image.open(f'images/{name}.png').convert('RGBA')
    white = Image.new('RGBA', img.size, (255, 255, 255, 255))
    return np.array(Image.alpha_composite(white, img).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))

poke_sub = poke[poke['Type1'].isin(CLASSES)].reset_index(drop=True)
X_img = np.stack([load_image(n) for n in poke_sub['Name']])
y_img = poke_sub['Type1'].map({c: i for i, c in enumerate(CLASSES)}).values

print('X_img:', X_img.shape)
print(poke_sub['Type1'].value_counts().reindex(CLASSES).to_string())

In [ ]:
rng = np.random.default_rng(42)
n_show = 8

fig, axes = plt.subplots(len(CLASSES), n_show, figsize=(n_show * 1.6, len(CLASSES) * 1.8))

for row, cls in enumerate(CLASSES):
    idx = rng.choice(np.where(y_img == row)[0], n_show, replace=False)
    for col, i in enumerate(idx):
        ax = axes[row, col]
        ax.imshow(X_img[i])
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(cls, fontsize=11)

fig.suptitle('Examples of each class')
plt.tight_layout()
plt.show()

## 2.2 Розділення та аугментація

In [ ]:
idx_all = np.arange(len(y_img))
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=42, stratify=y_img)
idx_train, idx_val = train_test_split(idx_train, test_size=0.25, random_state=42, stratify=y_img[idx_train])

print('Train:', len(idx_train), '  Validation:', len(idx_val), '  Test:', len(idx_test))

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), fill=255),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class ImageDataset(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = idx, tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return self.tf(X_img[j]), int(y_img[j])


def image_loader(idx, train=False, batch_size=32):
    return DataLoader(ImageDataset(idx, train_tf if train else eval_tf),
                      batch_size=batch_size, shuffle=train)


img_train_dl = image_loader(idx_train, train=True)
img_val_dl = image_loader(idx_val)
img_test_dl = image_loader(idx_test)

## 2.3 Проста CNN, навчена з нуля

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(42)
cnn = SimpleCNN(len(CLASSES))
print(cnn)
print('Trainable parameters:', count_params(cnn))

In [ ]:
hist_cnn = fit(cnn, img_train_dl, img_val_dl, epochs=80, lr=1e-3, patience=20, verbose=10)

In [ ]:
plot_history({'Simple CNN': hist_cnn}, 'Simple CNN learning curves')

## 2.4 Pretrained модель: ResNet-18

In [ ]:
def make_resnet(n_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model


torch.manual_seed(42)
resnet = make_resnet(len(CLASSES))
print('New classifier:', resnet.fc)
print('Trainable parameters:', count_params(resnet))

In [ ]:
hist_resnet = fit(resnet, img_train_dl, img_val_dl, epochs=40, lr=1e-4, patience=10, verbose=5)

## 2.5 Порівняння CNN з нуля та pretrained

In [ ]:
cnn_histories = {'Simple CNN': hist_cnn, 'ResNet-18 (pretrained)': hist_resnet}

plot_history(cnn_histories, 'Simple CNN vs pretrained ResNet-18')
print(history_summary(cnn_histories).round(3).T)

In [ ]:
def img_row(name, model):
    p, _, y_true = predict(model, img_test_dl)
    return {'Model': name,
            'Accuracy': accuracy_score(y_true, p),
            'Macro F1': f1_score(y_true, p, average='macro'),
            'Parameters': count_params(model)}, p

img_rows, img_preds = [], {}
for name, model in [('Simple CNN', cnn), ('ResNet-18 (pretrained)', resnet)]:
    row, img_preds[name] = img_row(name, model)
    img_rows.append(row)

y_img_test = y_img[idx_test]
print(pd.DataFrame(img_rows).set_index('Model').round(4).to_string())
print()
for name, p in img_preds.items():
    print(name)
    print(classification_report(y_img_test, p, target_names=CLASSES, digits=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ax, (name, p) in zip(axes, img_preds.items()):
    sns.heatmap(confusion_matrix(y_img_test, p), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

fig.suptitle('Confusion matrices (test set)')
plt.tight_layout()
plt.show()

## 2.6 Вплив обсягу тренувальних даних

In [ ]:
fractions = [0.25, 0.5, 0.75, 1.0]
volume = []

for frac in fractions:
    if frac < 1.0:
        idx_part = train_test_split(idx_train, train_size=frac, random_state=42,
                                    stratify=y_img[idx_train])[0]
    else:
        idx_part = idx_train
    part_dl = image_loader(idx_part, train=True)

    for name, make, epochs, lr, patience in [('Simple CNN', lambda: SimpleCNN(len(CLASSES)), 80, 1e-3, 20),
                                             ('ResNet-18 (pretrained)', lambda: make_resnet(len(CLASSES)), 40, 1e-4, 10)]:
        torch.manual_seed(42)
        model = make()
        h = fit(model, part_dl, img_val_dl, epochs=epochs, lr=lr, patience=patience, verbose=0)
        p, _, y_true = predict(model, img_test_dl)
        volume.append({'Fraction': frac, 'Train images': len(idx_part), 'Model': name,
                       'Accuracy': accuracy_score(y_true, p),
                       'Macro F1': f1_score(y_true, p, average='macro'),
                       'Best epoch': h['val_loss'].idxmin()})
        print(f'{frac:.0%} ({len(idx_part)} images)  {name:24s}  accuracy {volume[-1]["Accuracy"]:.3f}')

volume = pd.DataFrame(volume)

In [ ]:
print(volume.pivot(index='Train images', columns='Model', values=['Accuracy', 'Macro F1']).round(3).to_string())

plt.figure(figsize=(7, 4.2))
for name, color in [('Simple CNN', '#DD8452'), ('ResNet-18 (pretrained)', '#4C72B0')]:
    part = volume[volume['Model'] == name]
    plt.plot(part['Train images'], part['Accuracy'], marker='o', color=color, label=name)

plt.axhline(1 / len(CLASSES), color='gray', linestyle=':', label='random guess')
plt.xlabel('Number of training images')
plt.ylabel('Accuracy on test set')
plt.title('Accuracy vs training set size')
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Висновок

Для класифікації взято 4 типи покемонів (285 зображень). Обидві моделі дали однакову якість: проста CNN — accuracy 0.72, macro F1 0.66; ResNet-18 — 0.70 і 0.66 (якщо завжди відповідати «Water», accuracy буде 0.40). Обидві добре розпізнають Fire і Grass, а Electric майже не впізнають (recall 0.25): тип покемона не завжди видно з картинки, а тест із 57 зображень дуже малий — одна помилка змінює accuracy на 1.75 %.

Головна різниця — у процесі навчання. ResNet-18 вже має готові ознаки з ImageNet і досягає найкращого validation loss на 7-й епосі; далі train loss падає до нуля, а validation loss росте — модель з 11 млн параметрів швидко запам'ятовує 171 зображення. CNN з нуля навчається повільно (найкраща епоха 62), її validation loss дуже нестабільний. Одна епоха ResNet у 4 рази довша, але загальний час навчання майже однаковий (≈18–20 с).

Зі зменшенням вибірки до 42 зображень CNN втрачає небагато (0.72 -> 0.67), а ResNet — більше (0.70 -> 0.58). Pretrained модель тут не дає переваги, бо мультяшні спрайти на білому фоні сильно відрізняються від фотографій ImageNet, а для розрізнення типів вистачає кольору, який проста CNN вивчає й сама.

# Завдання 3. Нейронні мережі для класифікації текстів

**Датасет:** IMDB Movie Reviews (ЛР2)

## 3.1 Завантаження даних

In [ ]:
reviews = pd.read_csv('../IMDB_Dataset.csv').drop_duplicates('review').reset_index(drop=True)
reviews['label'] = (reviews['sentiment'] == 'positive').astype(int)

print('Size:', reviews.shape)
print(reviews['sentiment'].value_counts().to_string())

## 3.2 Токенізація

In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace("n't", ' not')
    return re.findall(r'[a-z]+', text)

reviews['tokens'] = reviews['review'].map(tokenize)
lengths = reviews['tokens'].str.len()

print(reviews['tokens'][1][:30])
print()
print(lengths.describe(percentiles=[0.5, 0.75, 0.9]).round(0).to_string())

In [ ]:
MAX_LEN = 200

plt.figure(figsize=(8, 3.8))
plt.hist(lengths.clip(upper=1000), bins=60, color='#4C72B0', alpha=0.8)
plt.axvline(MAX_LEN, color='#C44E52', linestyle='--', label=f'MAX_LEN = {MAX_LEN}')
plt.xlabel('Tokens per review (clipped at 1000)')
plt.ylabel('Reviews')
plt.title('Review length distribution')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Reviews not longer than {MAX_LEN} tokens: {(lengths <= MAX_LEN).mean():.1%}')

## 3.3 Словник, індекси та padding

In [ ]:
txt_train, txt_test = train_test_split(reviews.index, test_size=0.2, random_state=42,
                                       stratify=reviews['label'])
txt_train, txt_val = train_test_split(txt_train, test_size=0.1, random_state=42,
                                      stratify=reviews.loc[txt_train, 'label'])

MAX_VOCAB = 20000
word_counts = Counter(w for tokens in reviews.loc[txt_train, 'tokens'] for w in tokens)

itos = ['<pad>', '<unk>'] + [w for w, _ in word_counts.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}

covered = sum(word_counts[w] for w in itos[2:]) / sum(word_counts.values())
print('Unique words in train:', len(word_counts))
print('Vocabulary size:', len(itos))
print(f'Share of train tokens covered by vocabulary: {covered:.1%}')

In [ ]:
def encode(tokens):
    ids = [stoi.get(w, 1) for w in tokens][-MAX_LEN:]
    return ids + [0] * (MAX_LEN - len(ids))

def encode_split(idx):
    X = np.array([encode(t) for t in reviews.loc[idx, 'tokens']], dtype=np.int64)
    return X, reviews.loc[idx, 'label'].values

X_seq_train, y_seq_train = encode_split(txt_train)
X_seq_val, y_seq_val = encode_split(txt_val)
X_seq_test, y_seq_test = encode_split(txt_test)

print('Train:', X_seq_train.shape, '  Validation:', X_seq_val.shape, '  Test:', X_seq_test.shape)
print()
print('Example:', X_seq_train[0][-20:])
print('Decoded:', ' '.join(itos[i] for i in X_seq_train[0][-20:]))

In [ ]:
seq_train_dl = to_loader(X_seq_train, y_seq_train, batch_size=128, shuffle=True)
seq_val_dl = to_loader(X_seq_val, y_seq_val, batch_size=512)
seq_test_dl = to_loader(X_seq_test, y_seq_test, batch_size=512)

## 3.4 Модель: Embedding + LSTM

In [ ]:
EMB_DIM = 100

class TextLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden=128, n_classes=2, embeddings=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if embeddings is not None:
            self.embedding.weight.data.copy_(torch.as_tensor(embeddings))
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        lengths = (x != 0).sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(x), lengths,
                                                   batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.fc(self.dropout(h[-1]))


print(TextLSTM(len(itos), EMB_DIM))

## 3.5 а) Embedding з випадковою ініціалізацією

In [ ]:
torch.manual_seed(42)
lstm_rand = TextLSTM(len(itos), EMB_DIM)
print('Trainable parameters:', count_params(lstm_rand))

hist_rand = fit(lstm_rand, seq_train_dl, seq_val_dl, epochs=8, lr=1e-3, patience=3)

## 3.6 б) Pretrained embeddings: GloVe

In [ ]:
import zipfile
import urllib.request

GLOVE_DIR = '../glove'
GLOVE_FILE = f'{GLOVE_DIR}/glove.6B.{EMB_DIM}d.txt'

if not os.path.exists(GLOVE_FILE):
    os.makedirs(GLOVE_DIR, exist_ok=True)
    zip_path = f'{GLOVE_DIR}/glove.6B.zip'
    urllib.request.urlretrieve('https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip', zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extract(f'glove.6B.{EMB_DIM}d.txt', GLOVE_DIR)
    os.remove(zip_path)

glove = {}
with open(GLOVE_FILE, encoding='utf-8') as f:
    for line in f:
        word, *vec = line.rstrip().split(' ')
        if word in stoi:
            glove[word] = np.asarray(vec, dtype=np.float32)

rng = np.random.default_rng(42)
emb_matrix = rng.normal(0, 0.1, (len(itos), EMB_DIM)).astype(np.float32)
emb_matrix[0] = 0
for w, vec in glove.items():
    emb_matrix[stoi[w]] = vec

print(f'Words found in GloVe: {len(glove)} of {len(itos)} ({len(glove) / len(itos):.1%})')
print('Missing examples:', [w for w in itos[2:] if w not in glove][:15])

In [ ]:
def nearest(word, k=6):
    v = emb_matrix[stoi[word]]
    sims = emb_matrix @ v / (np.linalg.norm(emb_matrix, axis=1) * np.linalg.norm(v) + 1e-9)
    return [itos[i] for i in np.argsort(-sims)[1:k + 1]]

for w in ['excellent', 'terrible', 'boring', 'actor']:
    print(f'{w:10s}', nearest(w))

In [ ]:
torch.manual_seed(42)
lstm_glove = TextLSTM(len(itos), EMB_DIM, embeddings=emb_matrix)
print('Trainable parameters:', count_params(lstm_glove))

hist_glove = fit(lstm_glove, seq_train_dl, seq_val_dl, epochs=8, lr=1e-3, patience=3)

## 3.7 Порівняння

In [ ]:
txt_histories = {'LSTM + random embeddings': hist_rand, 'LSTM + GloVe': hist_glove}

plot_history(txt_histories, 'LSTM learning curves')
print(history_summary(txt_histories).round(4).T)

In [ ]:
txt_rows, txt_preds = [], {}

for name, model in [('LSTM + random embeddings', lstm_rand), ('LSTM + GloVe', lstm_glove)]:
    p, probs, y_true = predict(model, seq_test_dl)
    txt_preds[name] = p
    txt_rows.append({'Model': name,
                     'Accuracy': accuracy_score(y_true, p),
                     'Precision': precision_score(y_true, p),
                     'Recall': recall_score(y_true, p),
                     'F1': f1_score(y_true, p),
                     'ROC-AUC': roc_auc_score(y_true, probs[:, 1])})

print(pd.DataFrame(txt_rows).set_index('Model').round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (name, p) in zip(axes, txt_preds.items()):
    sns.heatmap(confusion_matrix(y_seq_test, p), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['negative', 'positive'], yticklabels=['negative', 'positive'], ax=ax)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

fig.suptitle('Confusion matrices (test set)')
plt.tight_layout()
plt.show()

### Висновок

GloVe покриває 99.5 % словника, і схожі за змістом слова мають близькі вектори (`excellent` -> `terrific`, `superb`; `terrible` -> `horrible`, `awful`). Тому модель з GloVe навчається швидше і досягає кращої якості: accuracy 0.882, F1 0.878, ROC-AUC 0.951 проти 0.862, 0.862 і 0.938 з випадковими embeddings. Уже після першої епохи вона мала val accuracy 0.81 (випадкова — 0.76), а найкращий validation loss (0.302) отримала на 3-й епосі, тоді як випадкова — гірший (0.335) лише на 5-й.

Обидві моделі перенавчаються після кількох епох: train loss і далі падає, а validation loss росте, особливо в GloVe (0.30 → 0.42 за три епохи), тому early stopping повертає найкращі ваги. Випадкова модель навчалась менш стабільно: на 2-й епосі validation loss зріс.

Обидві LSTM поступаються TF-IDF + логістичній регресії з ЛР2 (F1 0.910): відгуки обрізано до 200 слів (повністю вміщується лише 58 %), а 35 тис. відгуків замало, щоб LSTM вивчила більше, ніж лінійна модель на біграмах.